# Validacion y calidad de datos

## Objetivo

Revisar la estructura, completitud y hallazgos principales de los datasets raw
que alimentan el analisis academico 2024.

Este notebook es descriptivo. No expone registros individuales y no utiliza
variables de docencia como eje analitico.


In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

OUTPUTS = ROOT / "outputs"
TABLES = OUTPUTS / "tables"
FIGURES = OUTPUTS / "figures"
REPORTS = OUTPUTS / "reports"

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)


## Resumen de datasets revisados

| dataset | filas | columnas | valores_vacios | porcentaje_vacios_maximo |
| --- | --- | --- | --- | --- |
| actas_notas | 3468 | 12 | 0 | 0.00% |
| asignaciones | 2879 | 7 | 2726 | 94.69% |
| calendario_academico | 4 | 8 | 4 | 50.00% |
| catalogo_cursos | 90 | 7 | 0 | 0.00% |
| configuracion_cursos | 90 | 12 | 0 | 0.00% |
| estudiantes_inscritos | 650 | 7 | 0 | 0.00% |
| inscritos_sistemas_anual | 3 | 7 | 0 | 0.00% |
| malla_prerrequisitos | 136 | 8 | 32 | 11.76% |
| oferta_academica | 205 | 19 | 188 | 73.66% |
| parametros_academicos | 6 | 5 | 0 | 0.00% |

Interpretacion: los 10 datasets raw tienen estructura reconocible y fueron
perfilados antes de iniciar limpieza y construccion de variables. Los datasets
02, 03 y 09 se tratan como provistos por el Departamento de Computo.

Conclusion: el insumo base es suficiente para continuar con ETL, siempre que
los hallazgos de calidad se resuelvan o documenten antes de modelar.


In [ ]:
perfil = pd.read_csv(TABLES / "01_dataset_profile.csv")
resumen_datasets = (
    perfil.groupby("dataset", as_index=False)
    .agg(filas=("filas", "max"), columnas=("columna", "nunique"), valores_vacios=("valores_vacios", "sum"))
    .sort_values("dataset")
)
resumen_datasets


## Hallazgos de calidad

| severidad | dataset | regla | columna | hallazgo | cantidad |
| --- | --- | --- | --- | --- | --- |
| alto | oferta_academica | duplicados_llave | periodo_academico + codigo_curso + seccion | Existen registros con llave principal duplicada. | 10 |
| medio | oferta_academica | nulos_criticos | jornada | La columna requerida tiene valores vacios no esperados. | 5 |
| medio | oferta_academica | nulos_criticos | horario | La columna requerida tiene valores vacios no esperados. | 5 |
| info | actas_notas | valores_especiales | nota_total | Conteo de valor especial `EQ`. | 11 |
| info | actas_notas | valores_especiales | resultado | Conteo de valor especial `NSP`. | 113 |
| info | asignaciones | valores_especiales | estado_asignacion | Conteo de valor especial `desasignado`. | 153 |

Interpretacion: los hallazgos mas relevantes se concentran en duplicados de
oferta academica y valores vacios de atributos operativos. Tambien se conservan
valores especiales como `NSP`, `EQ` y `desasignado`, porque tienen significado
academico o administrativo para fases posteriores.

Conclusion: la calidad inicial no bloquea el analisis, pero requiere decisiones
explicitas de limpieza y conservacion de valores especiales.


In [ ]:
hallazgos = pd.read_csv(TABLES / "01_data_quality_findings.csv")
hallazgos[["severidad", "dataset", "regla", "columna", "hallazgo", "cantidad"]]


## Hallazgos obtenidos

- Se confirma la existencia de los 10 datasets raw esperados.
- El catalogo contiene 90 cursos vigentes, que luego funcionan como base para
  medir oferta y cursos no ofertados por periodo.
- Los registros especiales se conservan para evitar perder informacion sobre
  equivalencias, no presentados y desasignaciones.
- La oferta academica presenta duplicados de llave que se resuelven en la fase
  de ETL, dejando trazabilidad de la decision.

## Conclusiones del notebook

La validacion inicial establece una base confiable para el analisis porque
identifica estructura, completitud y reglas de calidad antes de transformar los
datos. Los hallazgos no se ocultan: se documentan y se conectan con decisiones
posteriores de ETL.

## Conclusiones generales

El proyecto parte de datos institucionales suficientes para analizar oferta,
asignaciones y resultados de 2024. La lectura debe mantenerse descriptiva,
agregada y metodologicamente prudente, sin inferir causalidad ni exponer casos
individuales.
